# 🇺🇸 A Century of American Carbon
### A data story, built with the `viz_lib` library

Run each cell top to bottom (`Shift+Enter`). Every chart is produced by a function from the **`viz_lib`** library, and every dataset ships **inside** the installed package (`viz_lib.load_dataset(...)`).

**Story:** global context → by fuel source → the US over time.

## Setup
Run once. Clones the project, installs it (`pip install -e`), and imports the library. The datasets come bundled with the package — no separate download.

> If the repository is private, make it public for the demo or add a GitHub token to the clone URL.

In [ ]:
import os, sys
REPO = 'visualisation-lib'
if not os.path.exists(REPO):
    !git clone -q -b claude/viz-lib-plot-requirements-dbjbc0 https://github.com/rinikhaneja/visualisation-lib.git
!pip -q install -e {REPO}                       # installs viz_lib + bundled data

import pandas as pd
import matplotlib.pyplot as plt
import viz_lib
from viz_lib import ranked_bar, stacked_bar, stacked_area, load_dataset
from viz_lib.theme import apply_theme, series_color
apply_theme()
print('viz_lib ready ✓   bundled datasets:', viz_lib.datasets.available())

## 1 · Global context — who emits the most CO₂ per person?
Two peer groups on the **same color scale**; ▲/▼ show the change since 2014.

In [ ]:
df = load_dataset('co2_per_capita')
df = df.pivot_table(index='Entity', columns='Year', values='CO₂ emissions per capita')
df.columns = [f'y{c}' for c in df.columns]
df = df.reset_index()

OIL  = ['Qatar','Kuwait','Brunei','Bahrain','Trinidad and Tobago',
        'Saudi Arabia','United Arab Emirates','Oman']
ECON = ['United States','Russia','North America','China',
        'European Union (27)','World','United Kingdom','India']
vmax  = df[df.Entity.isin(OIL + ECON)].y2024.max()
world = df.loc[df.Entity == 'World', 'y2024'].iloc[0]
pick  = lambda names: df[df.Entity.isin(names)]

ranked_bar(pick(OIL), category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t', reference=world, reference_label='World average',
           title='A few small, oil-rich nations emit the most CO₂ per person',
           subtitle='Tonnes of CO₂ per person, 2024')
plt.show()

In [ ]:
ranked_bar(pick(ECON), category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t',
           title='Among big economies, the US still emits the most per person',
           subtitle='Tonnes of CO₂ per person, 2024 — same scale as the oil producers')
plt.show()

## 2 · What fuels drive it? — per-capita CO₂ by source
The same totals, now split into **coal / oil / gas / flaring / cement / other** — a replica of the Our World in Data chart.

In [ ]:
d = load_dataset('percapita_co2_by_source')
SEG = ['Coal','Oil','Gas','Flaring','Cement','Other industry']
OWID = {'Coal':'#6d6e70','Oil':'#c14b62','Gas':'#8c6bb1',
        'Flaring':'#c8a45c','Cement':'#2f8e7f','Other industry':'#6d8fc5'}
tonnes = lambda v: f'{v:.0f} t' if v >= 10 else f'{v:.1f} t'   # 6.4 t but 34 t

stacked_bar(d, category='Entity', segments=SEG, colors=OWID,
            value_fmt=tonnes, seg_label_min=0.05,
            title='Per capita CO₂ emissions by source, 2024',
            figsize=(11, 8))
plt.show()

## 3 · The US over time — CO₂ by fuel (the hero)
A century-long stacked area: **coal → oil → gas**, annotated with the events that shaped it.

In [ ]:
h = load_dataset('us_co2_by_fuel')
FUELS = ['Coal','Oil','Gas','Cement','Flaring','Other industry']
for f in FUELS:
    h[f] = pd.to_numeric(h[f], errors='coerce') / 1e9   # tonnes -> billion tonnes

EVENTS = [{'year':1932,'label':'1932\nGreat Depression','y':0.42},
          {'year':1945,'label':'1945\nWWII','y':0.72},
          {'year':1973,'label':'1973\nOil shock','y':0.9},
          {'year':2007,'label':'2007\nemissions peak','y':0.98},
          {'year':2020,'label':'2020\nCOVID','y':0.62}]

stacked_area(h, x='Year', series=FUELS, y_label='Billion tonnes CO₂ / year',
             title='Coal gave way to oil and gas',
             subtitle='US CO₂ emissions by fuel or industry, 1800–2024',
             events=EVENTS)
plt.show()

---
*Built with `viz_lib` — a small, importable plotting library with bundled datasets.*  
Data: Global Carbon Budget (2025) via Our World in Data (CC BY).